# Example 08 · Thinking in LangGraph

**Source:** [LangChain Docs — Thinking in LangGraph](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph)

---

## Objectives

By the end of this notebook you will be able to:

1. Map a real-world workflow (customer support email) to a LangGraph `StateGraph`
2. Design a `TypedDict` state that stores raw data shared across nodes
3. Implement the four node types: LLM, Data, Action, and User-Input nodes
4. Apply error-handling strategies: `RetryPolicy`, Command-loop recovery, `error_handler`
5. Wire up `interrupt()` for human-in-the-loop review before sending replies
6. Compile and run the full graph end-to-end

---

## Background

### The Mental Model: Workflow → Graph

Every agent task can be decomposed into a directed graph:

```
START
  └─► read_email
         └─► classify_intent ──► search_documentation ─┐
                             ──► bug_tracking           ├─► draft_response ──► human_review ──► send_reply ──► END
                             ──► human_review ──────────┘             └────────────────────────────────────────────► END
```

### Four Node Types

| Type | What it does | Examples |
|------|-------------|----------|
| **LLM node** | Understands, analyzes, generates text | `classify_intent`, `draft_response` |
| **Data node** | Reads/writes external systems | `search_documentation`, `lookup_customer_history` |
| **Action node** | Executes side effects | `send_reply`, `bug_tracking` |
| **User-input node** | Pauses for human decision | `human_review` |

### State Design Rule

> **Store raw data, not formatted text.**
> Good: `email_content: str`, `classification: EmailClassification`
> Bad: `formatted_prompt: str` (derivable from other fields)

### Error Handling Matrix

| Error type | Owner | Strategy | LangGraph API |
|------------|-------|----------|---------------|
| Transient (network, rate limit) | System | Auto-retry | `RetryPolicy` on `add_node` |
| LLM-recoverable (tool fail, parse) | LLM | Feed error back, loop | `Command(update={error}, goto=node)` |
| User-fixable (missing info) | Human | Pause for input | `interrupt()` |
| Exhausted retries | Developer | Compensate | `error_handler` on `add_node` |
| Unexpected | Developer | Surface & debug | `raise` |

Run cells **top to bottom**.

## Setup

In [ ]:
import sys
from pathlib import Path
from typing import TypedDict, Literal

# 添加项目根目录，使 common/ 包可被导入
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt, RetryPolicy
from langgraph.errors import NodeError
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

# 初始化 LLM 和 Langfuse 追踪
llm = create_llm()
_handler = create_langfuse_handler()

def make_config(thread_id: str = "s08", trace_name: str | None = None) -> dict:
    cfg = build_langfuse_config(_handler, session_id=thread_id, trace_name=trace_name or thread_id)
    # LangGraph 需要 configurable.thread_id 来启用 checkpointer
    cfg["configurable"] = {"thread_id": thread_id}
    return cfg

print("✓ Setup complete")

---

## Part 1 · Workflow Mapping & State Design

Before writing any code, map the business process to nodes.
Then design the state — the single dict shared by every node.

**Golden rule:** State fields must be either:
- Needed by **multiple** nodes, OR
- **Expensive** to re-compute

If neither is true, keep it inside the node function as a local variable.

In [ ]:
# ── Nested type for structured LLM output ────────────────────────────────────
class EmailClassification(TypedDict):
    # 邮件意图：问题/Bug/账单/功能请求/复杂问题
    intent: Literal["question", "bug", "billing", "feature", "complex"]
    # 紧急程度
    urgency: Literal["low", "medium", "high", "critical"]
    topic: str    # 主题关键词
    summary: str  # 一句话摘要

# ── Main graph state ─────────────────────────────────────────────────────────
class EmailAgentState(TypedDict):
    # ── 输入字段（图入口时设置） ──────────────────────
    email_content:    str
    sender_email:     str
    email_id:         str
    # ── LLM 节点写入，后续节点读取 ─────────────────────
    classification:   EmailClassification | None
    # ── 数据节点写入 ───────────────────────────────────
    search_results:   list[str] | None
    customer_history: dict | None
    # ── 行动节点写入 ───────────────────────────────────
    draft_response:   str | None
    # ── 消息历史（LLM 上下文） ──────────────────────────
    messages:         list | None

print("✓ State types defined")
print("EmailAgentState fields:", list(EmailAgentState.__annotations__.keys()))

---

## Part 2 · Node Implementations

Each node is a plain Python function: takes state, does one thing, returns updates.
We implement all four node types for the email support workflow.

### Step 2a — LLM nodes: `read_email` and `classify_intent`

In [ ]:
# ── LLM Node 1: Read Email ───────────────────────────────────────────────────
def read_email(state: EmailAgentState) -> dict:
    # 将原始邮件内容包装为 HumanMessage，初始化消息历史
    print(f"  [read_email] processing email from {state['sender_email']}")
    return {
        "messages": [HumanMessage(content=f"Processing email: {state['email_content'][:80]}...")]
    }

# ── LLM Node 2: Classify Intent ──────────────────────────────────────────────
def classify_intent(
    state: EmailAgentState,
) -> Command[Literal["search_documentation", "bug_tracking", "human_review", "draft_response"]]:
    # 使用结构化输出强制 LLM 返回 EmailClassification 格式
    structured_llm = llm.with_structured_output(EmailClassification)

    classification_prompt = (
        f"Analyze this customer email and classify it.\n"
        f"Email: {state['email_content']}\n"
        f"From: {state['sender_email']}\n"
        f"Return: intent (question/bug/billing/feature/complex), urgency (low/medium/high/critical), topic, summary."
    )

    classification = structured_llm.invoke(classification_prompt)
    intent   = classification["intent"]
    urgency  = classification["urgency"]

    # 路由逻辑：根据分类结果决定下一节点
    if intent == "billing" or urgency == "critical":
        goto = "human_review"           # 账单/紧急 → 人工审核
    elif intent in ("question", "feature"):
        goto = "search_documentation"   # 问题/功能 → 搜索文档
    elif intent == "bug":
        goto = "bug_tracking"           # Bug → 创建工单
    else:
        goto = "draft_response"         # 其他 → 直接起草回复

    print(f"  [classify_intent] intent={intent!r}, urgency={urgency!r} → {goto}")

    # Command 同时完成两件事：更新状态 + 指定下一节点
    return Command(update={"classification": classification}, goto=goto)

print("✓ LLM nodes defined")

### Step 2b — Data nodes: `search_documentation` and `bug_tracking`

In [ ]:
# ── Data Node 1: Search Documentation ───────────────────────────────────────
def search_documentation(
    state: EmailAgentState,
) -> Command[Literal["draft_response"]]:
    # 数据节点：读取外部知识库；RetryPolicy 处理暂时性网络故障
    classification = state.get("classification") or {}
    query = f"{classification.get('intent', '')} {classification.get('topic', '')}"

    # 模拟知识库查询（真实项目替换为 RAG 检索）
    kb = {
        "password": [
            "Reset via Settings > Security > Change Password",
            "Minimum 12 chars: upper, lower, numbers, symbols",
        ],
        "billing": [
            "Invoices available in Account > Billing",
            "Refunds processed within 5–7 business days",
        ],
        "feature": [
            "Feature requests tracked at feedback.example.com",
            "Upvoted requests prioritized in roadmap",
        ],
    }
    results = []
    for key, docs in kb.items():
        if key in query.lower():
            results.extend(docs)
    if not results:
        results = ["No specific documentation found. Using general guidelines."]

    print(f"  [search_documentation] found {len(results)} docs for: {query[:40]!r}")
    return Command(update={"search_results": results}, goto="draft_response")

# ── Data Node 2: Bug Tracking ─────────────────────────────────────────────────
def bug_tracking(
    state: EmailAgentState,
) -> Command[Literal["draft_response"]]:
    # 行动节点：创建 Bug 工单（带 RetryPolicy 保护）
    classification = state.get("classification") or {}
    ticket_id = f"BUG-{hash(state['email_content']) % 90000 + 10000}"

    print(f"  [bug_tracking] created ticket {ticket_id}")
    return Command(
        update={"search_results": [f"Bug ticket {ticket_id} created and assigned to engineering team."]},
        goto="draft_response",
    )

print("✓ Data nodes defined")

### Step 2c — Action node: `draft_response` and `send_reply`

In [ ]:
# ── Action Node 1: Draft Response ────────────────────────────────────────────
def draft_response(
    state: EmailAgentState,
) -> Command[Literal["human_review", "send_reply"]]:
    # 综合分类结果和搜索结果，用 LLM 起草回复
    classification = state.get("classification") or {}
    intent  = classification.get("intent", "unknown")
    urgency = classification.get("urgency", "medium")

    # 组装上下文：搜索结果 + 客户历史
    context_parts = []
    if state.get("search_results"):
        docs = chr(10).join(f"- {d}" for d in state["search_results"])
        context_parts.append(f"Relevant documentation:\n{docs}")
    if state.get("customer_history"):
        tier = state["customer_history"].get("tier", "standard")
        context_parts.append(f"Customer tier: {tier}")

    draft_prompt = (
        f"Draft a professional reply to this customer email.\n"
        f"Email: {state['email_content']}\n"
        f"Intent: {intent}, Urgency: {urgency}\n"
        + (chr(10).join(context_parts) + chr(10) if context_parts else "")
        + "Be concise, helpful, and professional."
    )

    response = llm.invoke(draft_prompt)

    # 高优先级或复杂问题需要人工审核
    needs_review = (urgency in ("high", "critical") or intent == "complex")
    goto = "human_review" if needs_review else "send_reply"

    print(f"  [draft_response] needs_review={needs_review} → {goto}")
    return Command(update={"draft_response": response.content}, goto=goto)

# ── Action Node 2: Send Reply ─────────────────────────────────────────────────
def send_reply(state: EmailAgentState) -> dict:
    # 最终行动节点：发送邮件（真实项目替换为 email_service.send(...)）
    draft = state.get("draft_response", "(no draft)")
    print(f"  [send_reply] ✉ Sent to {state['sender_email']}: {draft[:80]}...")
    return {}

print("✓ Action nodes defined")

### Step 2d — User-input node: `human_review`

In [ ]:
# ── User-Input Node: Human Review ────────────────────────────────────────────
def human_review(
    state: EmailAgentState,
) -> Command[Literal["send_reply", "__end__"]]:
    # interrupt() 暂停图执行，等待人工决定
    # 图的完整状态已被 checkpointer 持久化，可以稍后恢复
    classification = state.get("classification") or {}

    human_decision = interrupt({
        "email_id":       state.get("email_id", ""),
        "original_email": state.get("email_content", ""),
        "draft_response": state.get("draft_response", ""),
        "urgency":        classification.get("urgency"),
        "intent":         classification.get("intent"),
        "action":         "Please review and approve/edit/reject this response",
    })

    if human_decision.get("approved"):
        # 人工可能修改了草稿，使用编辑后的版本（若无修改则保留原草稿）
        edited = human_decision.get("edited_response") or state.get("draft_response", "")
        print(f"  [human_review] ✓ Approved")
        return Command(update={"draft_response": edited}, goto="send_reply")
    else:
        print(f"  [human_review] ✗ Rejected — discarding")
        return Command(update={}, goto=END)

print("✓ User-input node defined")

---

## Part 3 · Error Handling Patterns

LangGraph provides three complementary mechanisms to make graphs resilient:

| Mechanism | Where configured | Handles |
|-----------|-----------------|--------|
| `RetryPolicy` | `add_node(..., retry_policy=...)` | Transient failures (network, rate limits) |
| `Command` loop-back | Inside the node function | LLM-recoverable errors |
| `error_handler` | `add_node(..., error_handler=...)` | Compensation after exhausted retries |

In [ ]:
# ── Pattern 1: RetryPolicy for transient failures ────────────────────────────
# search_documentation 使用指数退避重试，最多 3 次
search_retry_policy = RetryPolicy(
    max_attempts=3,
    initial_interval=1.0,   # 第 1 次重试等待 1 秒
    backoff_factor=2.0,      # 指数退避：1s → 2s → 4s
    max_interval=10.0,       # 最长等待不超过 10 秒
)
print("RetryPolicy:", search_retry_policy)

# ── Pattern 2: Command-loop for LLM-recoverable errors ───────────────────────
# 当工具调用失败时，把错误信息放回 state 让 LLM 再试一次
def execute_tool_safe(
    state: dict,
) -> Command[Literal["agent", "execute_tool"]]:
    # 模拟 LLM 可恢复的工具错误处理模式
    try:
        result = f"tool result for: {state.get('tool_call', 'unknown')}"
        return Command(update={"tool_result": result}, goto="agent")
    except Exception as e:
        # 将错误信息写入状态，路由回 agent 节点让 LLM 调整策略
        return Command(update={"tool_result": f"Tool error: {e}"}, goto="agent")

# ── Pattern 3: error_handler for compensation after exhausted retries ─────────
def payment_node(state: dict) -> dict:
    # 模拟可能失败的支付操作
    raise ConnectionError("Payment gateway unreachable")

def payment_error_handler(state: dict, error: NodeError) -> Command[Literal["finalize"]]:
    # 所有重试耗尽后执行的补偿逻辑（如退款、记录失败等）
    print(f"  [error_handler] compensating after: {error.node_id} failed")
    return Command(
        update={"status": f"compensated: {str(error)}"},
        goto="finalize",
    )

print("✓ Error handling patterns defined")

---

## Part 4 · Graph Compilation & End-to-End Run

Wire all nodes together, attach a `MemorySaver` checkpointer for interrupt/resume,
then run two scenarios.

**Checkpointer rule:** `interrupt()` requires a checkpointer — it saves the full
graph state so execution can resume exactly where it left off, even days later.

In [ ]:
# ── Build and compile the full graph ─────────────────────────────────────────
workflow = StateGraph(EmailAgentState)

# 注册节点；search_documentation 附加 RetryPolicy 防止暂时性故障
workflow.add_node("read_email",      read_email)
workflow.add_node("classify_intent", classify_intent)
workflow.add_node(
    "search_documentation",
    search_documentation,
    retry_policy=RetryPolicy(max_attempts=3, initial_interval=1.0),
)
workflow.add_node("bug_tracking",   bug_tracking)
workflow.add_node("draft_response", draft_response)
workflow.add_node("human_review",   human_review)
workflow.add_node("send_reply",     send_reply)

# 线性边：入口 → read_email → classify_intent
# classify_intent 使用 Command(goto=...) 动态路由，无需 add_conditional_edges
workflow.add_edge(START, "read_email")
workflow.add_edge("read_email", "classify_intent")
workflow.add_edge("send_reply", END)

# 使用 MemorySaver 作为 checkpointer（interrupt() 必须有 checkpointer）
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

print("✓ Graph compiled")
print("Nodes:", list(app.graph.nodes))

### Step 4a — Scenario A: Simple question (no human review needed)

In [ ]:
# Scenario A: 简单问题，路由到 search_documentation → draft_response → send_reply
email_a = {
    "email_content": "Hi, I forgot my password and cannot log in. How do I reset it?",
    "sender_email":  "user@example.com",
    "email_id":      "EMAIL-001",
    "classification": None, "search_results": None,
    "customer_history": None, "draft_response": None, "messages": None,
}

print("=== Scenario A: Password Reset Question ===")
result_a = app.invoke(email_a, config=make_config("s08-scenA"))
print(f"\nFinal draft: {result_a['draft_response'][:200]}...")

### Step 4b — Scenario B: Critical billing issue (triggers human review)

In [ ]:
# Scenario B: 账单问题，urgency=critical → 路由到 human_review → interrupt()
email_b = {
    "email_content": "I was charged TWICE for my subscription this month! This is urgent, please fix immediately!",
    "sender_email":  "angry@example.com",
    "email_id":      "EMAIL-002",
    "classification": None, "search_results": None,
    "customer_history": None, "draft_response": None, "messages": None,
}

cfg_b = make_config("s08-scenB")

print("=== Scenario B: Critical Billing Issue ===" )
result_b = app.invoke(email_b, config=cfg_b)

if result_b.get("__interrupt__"):
    pending = result_b["__interrupt__"][0].value
    print(f"\n⚠  Human review required!")
    print(f"   Intent:  {pending['intent']}")
    print(f"   Urgency: {pending['urgency']}")
    print(f"   Draft:   {pending['draft_response'][:120]}...")
else:
    print("No interrupt — email sent directly.")

In [ ]:
# ── Resume with human approval ────────────────────────────────────────────────
# interrupt() 保存了完整的图状态；向相同 thread_id 发送 Command(resume=...) 恢复执行
final_b = app.invoke(
    Command(resume={
        "approved": True,
        "edited_response": (
            "Dear customer, we sincerely apologize for the double charge. "
            "We have initiated an immediate full refund to your account. "
            "Please allow 3-5 business days for processing."
        ),
    }),
    config=cfg_b,   # 同一 thread_id → 从中断处恢复
)
print(f"\n✓ Email sent after human approval")
print(f"Final reply: {final_b.get('draft_response', '(none)')}...")

---

## Part 5 · Key Design Principles

### Node Granularity — Why Separate Nodes?

```
✓  One node per external service  → independent RetryPolicy per service
✓  One node per LLM call          → inspect classification before acting
✓  One node per side effect       → clear checkpoint boundary before/after
✓  One node per human decision    → interrupt() pauses exactly there
```

**Checkpointing insight:** LangGraph saves state at every node boundary.
Finer nodes = more checkpoints = less work to repeat after failure.

### State Management Rules

| Rule | Rationale |
|------|-----------|
| Store raw data, not formatted prompts | Node functions build prompts from state — avoids duplication |
| Include data used by ≥2 nodes | Single-use data stays as local variables |
| Include expensive-to-recompute data | Search results, external API responses |

### `interrupt()` Golden Rule

> When combining `interrupt()` with other state updates, **call `interrupt()` first**.
> Any state written before `interrupt()` is saved; any written after is not yet persisted.

```python
def human_review(state):
    decision = interrupt({"draft": state["draft_response"]})  # ← pause here
    return Command(update={...}, goto=...)                     # ← resume here
```

---

## Summary

| Concept | API | Notes |
|---------|-----|-------|
| Graph state | `TypedDict` | Raw data shared across all nodes |
| LLM node with routing | `Command(update={...}, goto="next")` | Update state + set next node in one call |
| Structured LLM output | `llm.with_structured_output(TypedDict)` | Returns validated dict matching the schema |
| Transient error retry | `RetryPolicy(max_attempts=N, ...)` | Passed to `add_node` |
| Compensation handler | `error_handler=fn` in `add_node` | Runs after all retries exhausted |
| Human pause | `interrupt({payload})` | Saves full state; resumes with `Command(resume=...)` |
| Checkpointer | `MemorySaver()` | Required for `interrupt()` and multi-turn state |

### Key Takeaways

1. **Map workflow first** — draw the flowchart before writing any code
2. **State = shared memory** — store raw data only; derive formatted text inside nodes
3. **`Command` is dual-purpose** — update state AND route to the next node in one return value
4. **Match error strategy to error type** — retry transient, loop LLM-recoverable, pause user-fixable
5. **Checkpointing is free** — more nodes = more resilience with no performance cost

In [ ]:
print(f'Traces available at: {get_langfuse_host()}')